In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.float_format", lambda x: "%.4f" % x)
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

with open("../../data/network_anomaly/meta.json") as f:
    meta = json.load(f)

train_df = pd.read_csv("../../data/network_anomaly/train_cleaned.csv")
test_df = pd.read_csv("../../data/network_anomaly/test_cleaned.csv")

print("Data loaded")
print(f"Train: {train_df.shape} | Test: {test_df.shape}")
print(f"Attack rate train: {train_df['is_attack'].mean()*100:.2f}%")
print(f"Attack rate test : {test_df['is_attack'].mean()*100:.2f}%")

Data loaded
Train: (125973, 44) | Test: (22544, 44)
Attack rate train: 46.54%
Attack rate test : 56.92%


In [2]:
# Encode Categorical Features
print("ENCODING CATEGORICAL FEATURES")
print()

protocol_map = {"tcp": 0, "udp": 1, "icmp": 2}
train_df["protocol_encoded"] = train_df["protocol_type"].map(protocol_map).fillna(3)
test_df["protocol_encoded"] = test_df["protocol_type"].map(protocol_map).fillna(3)

flag_map = {"SF": 0, "S0": 1, "REJ": 2, "RSTR": 3, "SH": 4,
            "RSTO": 5, "S1": 6, "RSTOS0": 7, "S3": 8, "S2": 9, "OTH": 10}
train_df["flag_encoded"] = train_df["flag"].map(flag_map).fillna(11)
test_df["flag_encoded"] = test_df["flag"].map(flag_map).fillna(11)

attack_rate_per_service = train_df.groupby("service")["is_attack"].mean()
train_df["service_attack_rate"] = train_df["service"].map(attack_rate_per_service).fillna(0.5)
test_df["service_attack_rate"] = test_df["service"].map(attack_rate_per_service).fillna(0.5)

service_count = train_df["service"].value_counts()
train_df["service_frequency"] = train_df["service"].map(service_count).fillna(0)
test_df["service_frequency"] = test_df["service"].map(service_count).fillna(0)

print("Categorical features encoded:")
print("  protocol_type  -> protocol_encoded (tcp=0, udp=1, icmp=2)")
print("  flag           -> flag_encoded (SF=0, S0=1, REJ=2, ...)")
print("  service        -> service_attack_rate (target encoding)")
print("  service        -> service_frequency (count encoding)")
print()

print("Service attack rate stats:")
print(f"  Min  : {train_df['service_attack_rate'].min():.4f}")
print(f"  Max  : {train_df['service_attack_rate'].max():.4f}")
print(f"  Mean : {train_df['service_attack_rate'].mean():.4f}")
print()

print("Top 5 highest-risk services:")
top_risk = attack_rate_per_service.sort_values(ascending=False).head(5)
for svc, rate in top_risk.items():
    count = service_count.get(svc, 0)
    print(f"  {svc:<15}: {rate*100:.1f}% attack rate ({count:,} samples)")

ENCODING CATEGORICAL FEATURES

Categorical features encoded:
  protocol_type  -> protocol_encoded (tcp=0, udp=1, icmp=2)
  flag           -> flag_encoded (SF=0, S0=1, REJ=2, ...)
  service        -> service_attack_rate (target encoding)
  service        -> service_frequency (count encoding)

Service attack rate stats:
  Min  : 0.0000
  Max  : 1.0000
  Mean : 0.4654

Top 5 highest-risk services:
  mtp            : 100.0% attack rate (439 samples)
  nntp           : 100.0% attack rate (296 samples)
  klogin         : 100.0% attack rate (433 samples)
  kshell         : 100.0% attack rate (299 samples)
  ldap           : 100.0% attack rate (410 samples)


In [3]:
# Network Traffic Features
print("ENGINEERING NETWORK TRAFFIC FEATURES")
print()

train_df["bytes_ratio"] = train_df["src_bytes"] / (train_df["dst_bytes"] + 1)
train_df["total_bytes"] = train_df["src_bytes"] + train_df["dst_bytes"]
train_df["bytes_log"] = np.log1p(train_df["total_bytes"])
train_df["src_bytes_log"] = np.log1p(train_df["src_bytes"])
train_df["dst_bytes_log"] = np.log1p(train_df["dst_bytes"])
train_df["is_zero_src"] = (train_df["src_bytes"] == 0).astype(int)
train_df["is_zero_dst"] = (train_df["dst_bytes"] == 0).astype(int)
train_df["high_src_bytes"] = (train_df["src_bytes"] > train_df["src_bytes"].quantile(0.95)).astype(int)

test_df["bytes_ratio"] = test_df["src_bytes"] / (test_df["dst_bytes"] + 1)
test_df["total_bytes"] = test_df["src_bytes"] + test_df["dst_bytes"]
test_df["bytes_log"] = np.log1p(test_df["total_bytes"])
test_df["src_bytes_log"] = np.log1p(test_df["src_bytes"])
test_df["dst_bytes_log"] = np.log1p(test_df["dst_bytes"])
test_df["is_zero_src"] = (test_df["src_bytes"] == 0).astype(int)
test_df["is_zero_dst"] = (test_df["dst_bytes"] == 0).astype(int)
test_df["high_src_bytes"] = (test_df["src_bytes"] > train_df["src_bytes"].quantile(0.95)).astype(int)

print("Byte-based features created:")
byte_features = ["bytes_ratio", "total_bytes", "bytes_log", "is_zero_src", "is_zero_dst", "high_src_bytes"]
overall = train_df["is_attack"].mean() * 100
for feat in byte_features:
    if train_df[feat].nunique() == 2:
        rate = train_df[train_df[feat] == 1]["is_attack"].mean() * 100
        lift = rate / overall
        print(f"  {feat:<20}: attack rate when=1 → {rate:.2f}% (lift {lift:.2f}x)")
    else:
        corr = abs(train_df[feat].corr(train_df["is_attack"]))
        print(f"  {feat:<20}: corr with attack = {corr:.4f}")

ENGINEERING NETWORK TRAFFIC FEATURES

Byte-based features created:
  bytes_ratio         : corr with attack = 0.0062
  total_bytes         : corr with attack = 0.0072
  bytes_log           : corr with attack = 0.7833
  is_zero_src         : attack rate when=1 → 93.09% (lift 2.00x)
  is_zero_dst         : attack rate when=1 → 83.88% (lift 1.80x)
  high_src_bytes      : attack rate when=1 → 17.47% (lift 0.38x)


In [4]:
# Connection Behavior Features
print("CONNECTION BEHAVIOR FEATURES")
print()

train_df["error_rate_sum"] = train_df["serror_rate"] + train_df["rerror_rate"]
train_df["srv_error_rate_sum"] = train_df["srv_serror_rate"] + train_df["srv_rerror_rate"]
train_df["high_error"] = (train_df["error_rate_sum"] > 0.5).astype(int)
train_df["connection_density"] = train_df["count"] * train_df["same_srv_rate"]
train_df["srv_diversity"] = 1 - train_df["same_srv_rate"]
train_df["host_srv_concentration"] = train_df["dst_host_srv_count"] * train_df["dst_host_same_srv_rate"]
train_df["suspicious_access"] = (
    (train_df["root_shell"] > 0) | (train_df["su_attempted"] > 0) |
    (train_df["num_compromised"] > 0) | (train_df["num_root"] > 0)
).astype(int)
train_df["file_activity"] = train_df["num_file_creations"] + train_df["num_shells"] + train_df["num_access_files"]
train_df["network_risk_score"] = (
    train_df["error_rate_sum"] * 0.25 +
    (1 - train_df["same_srv_rate"]) * 0.20 +
    train_df["service_attack_rate"] * 0.30 +
    (train_df["protocol_encoded"] == 2).astype(float) * 0.15 +
    train_df["suspicious_access"] * 0.10
)

for df in [test_df]:
    df["error_rate_sum"] = df["serror_rate"] + df["rerror_rate"]
    df["srv_error_rate_sum"] = df["srv_serror_rate"] + df["srv_rerror_rate"]
    df["high_error"] = (df["error_rate_sum"] > 0.5).astype(int)
    df["connection_density"] = df["count"] * df["same_srv_rate"]
    df["srv_diversity"] = 1 - df["same_srv_rate"]
    df["host_srv_concentration"] = df["dst_host_srv_count"] * df["dst_host_same_srv_rate"]
    df["suspicious_access"] = ((df["root_shell"] > 0) | (df["su_attempted"] > 0) |
                                (df["num_compromised"] > 0) | (df["num_root"] > 0)).astype(int)
    df["file_activity"] = df["num_file_creations"] + df["num_shells"] + df["num_access_files"]
    df["network_risk_score"] = (
        df["error_rate_sum"] * 0.25 +
        (1 - df["same_srv_rate"]) * 0.20 +
        df["service_attack_rate"] * 0.30 +
        (df["protocol_encoded"] == 2).astype(float) * 0.15 +
        df["suspicious_access"] * 0.10
    )

print("Behavior features created:")
behavior_feats = ["error_rate_sum", "high_error", "connection_density", "srv_diversity",
                  "suspicious_access", "file_activity", "network_risk_score"]
for feat in behavior_feats:
    corr = abs(train_df[feat].corr(train_df["is_attack"]))
    print(f"  {feat:<30}: corr with attack = {corr:.4f}")

print()
print(f"network_risk_score — attack mean  : {train_df[train_df['is_attack']==1]['network_risk_score'].mean():.4f}")
print(f"network_risk_score — normal mean  : {train_df[train_df['is_attack']==0]['network_risk_score'].mean():.4f}")

CONNECTION BEHAVIOR FEATURES

Behavior features created:
  error_rate_sum                : corr with attack = 0.7636
  high_error                    : corr with attack = 0.7661
  connection_density            : corr with attack = 0.0296
  srv_diversity                 : corr with attack = 0.7519
  suspicious_access             : corr with attack = 0.0091
  file_activity                 : corr with attack = 0.0280
  network_risk_score            : corr with attack = 0.9011

network_risk_score — attack mean  : 0.6171
network_risk_score — normal mean  : 0.0611


In [6]:
#  Feature Importance and Save
from sklearn.ensemble import RandomForestClassifier

print("FEATURE IMPORTANCE RANKING")
print()

drop_cols = ["label", "attack_category", "protocol_type", "service", "flag",
             "is_attack", "num_outbound_cmds"]
feature_cols = [c for c in train_df.columns if c not in drop_cols]
feature_cols = [c for c in feature_cols if train_df[c].dtype in ["float64","int64","float32","int32"]]

X = train_df[feature_cols].fillna(0)
y = train_df["is_attack"]

rf = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1, max_depth=8)
rf.fit(X, y)

importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
original_features = meta["numeric_cols"]

print("Top 20 features:")
for i, (feat, imp) in enumerate(importances.head(20).items(), 1):
    tag = "ENGINEERED" if feat not in original_features else "original"
    print(f"  {i:2}. {feat:<35} {imp:.6f}  [{tag}]")

eng_imp = importances[[f for f in importances.index if f not in original_features]].sum()
orig_imp = importances[[f for f in importances.index if f in original_features]].sum()
total = eng_imp + orig_imp
print()
print(f"Engineered importance: {eng_imp/total*100:.1f}%")
print(f"Original importance  : {orig_imp/total*100:.1f}%")

train_save = train_df.drop(columns=[c for c in drop_cols if c in train_df.columns]).fillna(0)
test_save = test_df.drop(columns=[c for c in drop_cols if c in test_df.columns]).fillna(0)

train_save.to_csv("../../data/network_anomaly/train_engineered.csv", index=False)
test_save.to_csv("../../data/network_anomaly/test_engineered.csv", index=False)

import json
meta["feature_cols"] = feature_cols
with open("../../data/network_anomaly/meta.json", "w") as f:
    json.dump(meta, f, indent=4)

print()
print(f"Train engineered: {train_save.shape}")
print(f"Test engineered : {test_save.shape}")


FEATURE IMPORTANCE RANKING

Top 20 features:
   1. bytes_log                           0.177848  [ENGINEERED]
   2. network_risk_score                  0.168052  [ENGINEERED]
   3. total_bytes                         0.087308  [ENGINEERED]
   4. service_attack_rate                 0.075918  [ENGINEERED]
   5. src_bytes                           0.074680  [original]
   6. is_zero_dst                         0.045578  [ENGINEERED]
   7. src_bytes_log                       0.035057  [ENGINEERED]
   8. dst_bytes                           0.033579  [original]
   9. high_error                          0.028055  [ENGINEERED]
  10. flag_encoded                        0.025974  [ENGINEERED]
  11. bytes_ratio                         0.022672  [ENGINEERED]
  12. protocol_encoded                    0.021762  [ENGINEERED]
  13. dst_bytes_log                       0.017732  [ENGINEERED]
  14. host_srv_concentration              0.015722  [ENGINEERED]
  15. same_srv_rate                       0.01450